# Aspect-aware sentiment analysis

This notebook preserves the original keyword-splitting and whole-review-classifier implementation as a baseline, including its known failure modes. The dashboard and improved workflow use shared `inference.py` preprocessing and helpers. The pretrained `yangheng/deberta-v3-base-absa-v1.1` model receives the review context and target aspect as a paired input and predicts only Negative, Neutral, or Positive. It has no separate no-opinion class; factual mentions are screened by a disclosed opinion-cue heuristic, and low-confidence or possible cross-product comparisons are marked uncertain.

The model card documents an MIT licence and CPU inference. Its first load downloads weights from Hugging Face. Review texts remain local. Manually authored sanity examples below are not evaluation data; real evaluation requires human-filled labels collected separately.

In [9]:
aspect_keywords = {
    "battery": ["battery", "battery life", "battery backup"],
    "sound": ["sound", "audio", "bass", "volume"],
    "camera": ["camera", "photo", "picture", "video quality"],
    "display": ["screen", "display", "brightness"],
    "performance": ["performance", "speed", "lag", "slow", "crashing"],
    "comfort": ["comfortable", "uncomfortable", "comfort", "ear pain"],
    "build_quality": ["build quality", "durable", "flimsy", "sturdy"],
    "price": ["price", "expensive", "cheap", "value for money"],
    "connectivity": ["bluetooth", "wifi", "wi-fi", "connection"],
    "charging": ["charging", "charger", "charges"]
}

In [10]:
import re

def detect_aspects(text):
    text = text.lower()
    detected = []

    for aspect, keywords in aspect_keywords.items():
        for keyword in keywords:
            pattern = r"\b" + re.escape(keyword) + r"\b"

            if re.search(pattern, text):
                detected.append(aspect)
                break

    return detected

In [11]:
review = "The sound is good, but the headphones are uncomfortable."

print(detect_aspects(review))

['sound', 'comfort']


In [12]:
def split_review(text):
    parts = re.split(
        r"[.!?;]+|\b(?:but|however|although|yet)\b",
        text,
        flags=re.IGNORECASE
    )

    return [part.strip() for part in parts if part.strip()]

In [13]:
review = "The sound is good, but the headphones are uncomfortable."

for part in split_review(review):
    print("Text:", part)
    print("Aspects:", detect_aspects(part))
    print()

Text: The sound is good,
Aspects: ['sound']

Text: the headphones are uncomfortable
Aspects: ['comfort']



In [14]:
import joblib

sentiment_model = joblib.load("sentiment_pipeline.pkl")

In [15]:
print("Kernel is working")


Kernel is working


In [16]:
import sys
print("Python:", sys.executable, flush=True)

import joblib
print("Joblib imported:", joblib.__version__, flush=True)

import sklearn
print("Scikit-learn imported:", sklearn.__version__, flush=True)

Python: c:\Users\vansh\OneDrive\Pictures\Desktop\sentimentanalysis\.venv\Scripts\python.exe
Joblib imported: 1.6.0
Scikit-learn imported: 1.9.1


In [17]:
from pathlib import Path

model_path = Path("sentiment_pipeline.pkl")

print("Looking at:", model_path.resolve())
print("File exists:", model_path.is_file())

if model_path.is_file():
    print("Size in MB:", round(model_path.stat().st_size / (1024 ** 2), 2))

Looking at: C:\Users\vansh\OneDrive\Pictures\Desktop\sentimentanalysis\sentiment_pipeline.pkl
File exists: True
Size in MB: 1.2


In [18]:
import subprocess
import sys

code = """
import joblib

model = joblib.load("sentiment_pipeline.pkl")
print("Model loaded successfully", flush=True)

prediction = model.predict(["the sound quality is excellent"])
print("Prediction:", prediction[0], flush=True)
"""

try:
    result = subprocess.run(
        [sys.executable, "-u", "-c", code],
        cwd=str(model_path.resolve().parent),
        capture_output=True,
        text=True,
        timeout=60
    )

    print("Exit code:", result.returncode)
    print("Output:", result.stdout)
    print("Errors:", result.stderr)

except subprocess.TimeoutExpired:
    print("Model loading or prediction exceeded 60 seconds.")

Exit code: 0
Output: Model loaded successfully
Prediction: positive

Errors: 


In [19]:
from inference import clean_text, load_sentiment_pipeline

sentiment_model = load_sentiment_pipeline()
print("Shared preprocessing and saved overall model are ready")

Model and cleaning function are ready


In [20]:
def analyze_review(text):
    results = []

    for part in split_review(text):
        aspects = detect_aspects(part)

        if not aspects:
            continue

        if len(aspects) > 1:
            sentiment = "needs_review"
        else:
            sentiment = sentiment_model.predict(
                [clean_text(part)]
            )[0]

        results.append({
            "text": part,
            "aspects": aspects,
            "sentiment": sentiment
        })

    return results

In [21]:
def get_pros_cons(text):
    output = {
        "pros": [],
        "cons": [],
        "neutral": [],
        "needs_review": []
    }

    for result in analyze_review(text):
        sentiment = result["sentiment"]

        if sentiment == "positive":
            output["pros"].append(result)

        elif sentiment == "negative":
            output["cons"].append(result)

        elif sentiment == "neutral":
            output["neutral"].append(result)

        else:
            output["needs_review"].append(result)

    return output

In [22]:
review = "The sound is excellent, but the headphones are uncomfortable."

results = get_pros_cons(review)

for section, items in results.items():
    print(f"\n{section.upper()}:")

    for item in items:
        print(item["aspects"], ":", item["text"])


PROS:
['sound'] : The sound is excellent,

CONS:
['comfort'] : the headphones are uncomfortable

NEUTRAL:

NEEDS_REVIEW:


In [23]:
test_reviews = [
    "The battery is excellent.",
    "The battery is not good.",
    "The headphones are comfortable.",
    "The headphones are not comfortable.",
    "Great sound and terrible battery.",
    "The screen is bright, but the battery drains quickly."
]

for review in test_reviews:
    print("\nREVIEW:", review)

    for result in analyze_review(review):
        print(
            result["aspects"],
            "→",
            result["sentiment"],
            "|",
            result["text"]
        )


REVIEW: The battery is excellent.
['battery'] → positive | The battery is excellent

REVIEW: The battery is not good.
['battery'] → negative | The battery is not good

REVIEW: The headphones are comfortable.
['comfort'] → positive | The headphones are comfortable

REVIEW: The headphones are not comfortable.
['comfort'] → negative | The headphones are not comfortable

REVIEW: Great sound and terrible battery.
['battery', 'sound'] → needs_review | Great sound and terrible battery

REVIEW: The screen is bright, but the battery drains quickly.
['display'] → neutral | The screen is bright,
['battery'] → neutral | the battery drains quickly


In [24]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("consumer_tech_reviews_9000.csv")

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, _ = next(
    splitter.split(df, groups=df["product_id"])
)

sample_reviews = df.iloc[train_idx].sample(
    n=5,
    random_state=42
)

for review in sample_reviews["review_text"]:
    print("\nREVIEW:", review)

    for result in analyze_review(review):
        print(
            result["aspects"],
            "→",
            result["sentiment"],
            "|",
            result["text"]
        )


REVIEW: The instructions simply show you ow to plug it in... that's all. The camera field of view is very small and the quality is poor. It is not 4K. It is as good as a camera from the 90s.
['camera'] → negative | The camera field of view is very small and the quality is poor
['camera'] → positive | It is as good as a camera from the 90s

REVIEW: We are changing over all of our AP's and have decided to go with Ubiquity brand. I have tried several of their AP's and really like the Unifi Mesh AP' It is a super simple install having come with the adapter to mount to a pole and an adapter to flush-mount to a flat surface. The range of these things is incredible and rivals even the Cisco AP's we have. These AP's come with a grommet where the Ethernet cable goes into it to keep out moisture and all are designed for POE installation. We have used both the POE Injector that came with the AP and other off-the-shelf injectors. So far, all work just fine with the AP's.

REVIEW: I have diagnosed

## Separate sanity checks (not evaluation)

These short hand-authored examples check expected behavior and evidence handling only. Their expected labels are not human ground truth and must not be used to report model performance.

In [25]:
import pandas as pd

aspect_checks = pd.DataFrame([
    {
        "text": "The battery drains quickly.",
        "aspect": "battery",
        "sanity_expectation_not_ground_truth": "negative"
    },
    {
        "text": "The sound is excellent.",
        "aspect": "sound",
        "sanity_expectation_not_ground_truth": "positive"
    },
    {
        "text": "The headphones are not comfortable.",
        "aspect": "comfort",
        "sanity_expectation_not_ground_truth": "negative"
    },
    {
        "text": "Before connecting the WiFi extender.",
        "aspect": "connectivity",
        "sanity_expectation_not_ground_truth": "no_opinion"
    },
    {
        "text": "It is sitting on the charger next to me.",
        "aspect": "charging",
        "sanity_expectation_not_ground_truth": "no_opinion"
    }
])

aspect_checks

,text,aspect,expected
0,The battery drains quickly.,battery,negative
1,The sound is excellent.,sound,positive
2,The headphones are not comfortable.,comfort,negative
3,Before connecting the WiFi extender.,connectivity,no_opinion
4,It is sitting on the charger next to me.,charging,no_opinion


In [27]:
from inference import analyze_aspect_evidence, load_absa_model

try:
    absa_model, absa_tokenizer = load_absa_model()
    aspect_method = "Pretrained DeBERTa ABSA: full clause + target-aspect pair"
except (ImportError, OSError, RuntimeError, ValueError, ConnectionError, TimeoutError) as error:
    absa_model, absa_tokenizer = None, None
    aspect_method = f"Explicit sentence-level baseline fallback: {error}"

sanity_reviews = [
    {"review_id": f"sanity-{index}", "review_text": row.text}
    for index, row in enumerate(aspect_checks.itertuples(), start=1)
]
sanity_results = analyze_aspect_evidence(
    sanity_reviews,
    sentiment_model,
    absa_model,
    absa_tokenizer,
)
print(aspect_method)
pd.DataFrame(sanity_results)[
    ["review_id", "review_text", "aspect", "sentiment", "confidence", "method"]
]

,text,aspect,expected,predicted
0,The battery drains quickly.,battery,negative,neutral
1,The sound is excellent.,sound,positive,positive
2,The headphones are not comfortable.,comfort,negative,negative
3,Before connecting the WiFi extender.,connectivity,no_opinion,positive
4,It is sitting on the charger next to me.,charging,no_opinion,negative


In [28]:
import importlib.util

for package in ["torch", "transformers"]:
    installed = importlib.util.find_spec(package) is not None
    print(f"{package}: {'installed' if installed else 'not installed'}")

torch: not installed
transformers: not installed


In [30]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

c:\Users\vansh\OneDrive\Pictures\Desktop\sentimentanalysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.14.1+cpu
Transformers: 5.20.1


## Human-annotated aspect evaluation

The next cell creates an annotation template from product IDs held out by a grouped split. It contains no model-generated labels. Annotators should use `positive`, `negative`, `neutral`, `mixed`, `no_opinion`, or `uncertain`, then save a completed copy as `aspect_annotations_completed.csv`. The pretrained classifier itself only predicts three labels; mixed, no-opinion, and uncertain are annotation/decision states, not pretrained model classes. The evaluation reports three-class metrics only for human-labeled positive/negative/neutral examples where the model makes a confident three-class prediction, and separately counts excluded and abstained cases.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from inference import detect_aspects, load_reviews

review_data = load_reviews()
_, heldout_indices = next(
    GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(
        review_data, groups=review_data["product_id"]
    )
)
heldout_reviews = review_data.iloc[heldout_indices]
annotation_rows = []
for row in heldout_reviews.itertuples():
    for aspect in detect_aspects(row.review_text):
        annotation_rows.append({
            "review_id": str(getattr(row, "review_id", "")),
            "product_id": str(row.product_id),
            "review_text": row.review_text,
            "aspect": aspect,
            "human_sentiment": "",
            "annotator_notes": "",
        })
annotation_template = pd.DataFrame(annotation_rows).drop_duplicates(
    ["review_id", "aspect"]
)
annotation_template = annotation_template.sample(
    n=min(100, len(annotation_template)), random_state=42
).reset_index(drop=True)
template_path = Path("aspect_annotations_template.csv")
if not template_path.exists():
    annotation_template.to_csv(template_path, index=False)
    print(f"Created {template_path} with {len(annotation_template)} unlabeled examples.")
else:
    print(f"Preserved existing {template_path}; no annotation file was overwritten.")
print("Fill human_sentiment manually, then save a copy as aspect_annotations_completed.csv.")
annotation_template.head()


In [ ]:
from pathlib import Path
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report
from inference import ABSA_CONFIDENCE_THRESHOLD, load_absa_model

completed_path = Path("aspect_annotations_completed.csv")
if not completed_path.exists():
    print("No completed human annotations found; no evaluation metrics were calculated.")
else:
    annotations = pd.read_csv(completed_path).dropna(subset=["human_sentiment"])
    required = {"review_text", "aspect", "human_sentiment"}
    missing = required - set(annotations.columns)
    if missing:
        raise ValueError(f"Completed annotation file is missing columns: {sorted(missing)}")
    valid_human_labels = {"positive", "negative", "neutral", "mixed", "no_opinion", "uncertain"}
    annotations["human_sentiment"] = annotations["human_sentiment"].str.strip().str.lower()
    invalid = set(annotations["human_sentiment"]) - valid_human_labels
    if invalid:
        raise ValueError(f"Unsupported human labels: {sorted(invalid)}")
    absa_model, absa_tokenizer = load_absa_model()
    inputs = absa_tokenizer(
        annotations["review_text"].astype(str).tolist(),
        annotations["aspect"].astype(str).tolist(),
        padding=True, truncation=True, max_length=512, return_tensors="pt"
    )
    absa_model.eval()
    with torch.inference_mode():
        probabilities = torch.softmax(absa_model(**inputs).logits, dim=-1)
    id_to_label = {int(key): value.lower() for key, value in absa_model.config.id2label.items()}
    confidence, class_ids = probabilities.max(dim=1)
    annotations["model_confidence"] = confidence.tolist()
    annotations["model_prediction"] = [
        id_to_label[int(class_id)] if score >= ABSA_CONFIDENCE_THRESHOLD else "uncertain"
        for class_id, score in zip(class_ids.tolist(), confidence.tolist())
    ]
    scored = annotations.loc[
        annotations["human_sentiment"].isin({"positive", "negative", "neutral"})
        & annotations["model_prediction"].isin({"positive", "negative", "neutral"})
    ]
    print(f"Human-annotated rows: {len(annotations)}")
    print(f"Rows scored in three-class metrics: {len(scored)}")
    print(f"Human mixed/no-opinion/uncertain: {len(annotations) - annotations['human_sentiment'].isin({'positive', 'negative', 'neutral'}).sum()}")
    print(f"Model abstentions (low-confidence): {(annotations['model_prediction'] == 'uncertain').sum()}")
    if scored.empty:
        print("No eligible rows for three-class metrics yet.")
    else:
        print("Accuracy:", accuracy_score(scored["human_sentiment"], scored["model_prediction"]))
        print(classification_report(
            scored["human_sentiment"], scored["model_prediction"],
            labels=["negative", "neutral", "positive"], zero_division=0
        ))
    annotations.to_csv("aspect_annotation_predictions.csv", index=False)
